In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt

In [ ]:
training_data = datasets.FashionMNIST(
  root="data",
  train=True,
  download=True,
  transform=ToTensor()
)
test_data = datasets.FashionMNIST(
  root="data",
  train=False,
  download=True,
  transform=ToTensor()
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 194kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.60MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 8.64MB/s]


In [ ]:
batch_size = 64
train_dataloader = DataLoader(training_data, batch_size)
test_dataloader = DataLoader(test_data, batch_size)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
class Net(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
      nn.Linear(28*28,512),
      nn.ReLU(),
      nn.Linear(512,512),
      nn.ReLU(),
      nn.Linear(512,10)
    )
  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits
model = Net().to(device)
print(model)

Net(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train(dataloader, model, loss_fn, optimizer):
  size = len(dataloader.dataset)
  model.train()

  for batch, (X,y) in enumerate(dataloader):
    X,y = X.to(device), y.to(device)

    pred = model(X)
    loss = loss_fn(pred,y)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if batch % 100 == 0:
      loss, current = loss.item(), batch * len(X)
      print(f"loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

def test(dataloader, model, loss_fn):
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  model.eval()
  test_loss, correct = 0,0

  with torch.inference_mode():
    for X,y in dataloader:
      X,y = X.to(device), y.to(device)
      pred = model(X)
      test_loss += loss_fn(pred, y).item()
      correct += (pred.argmax(1) == y).type(torch.float).sum().item()

  test_loss /= num_batches
  correct /= size
  print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
epochs = 10

for t in range(epochs):
  print(f"Epoch {t+1}\n-------------------------------")
  train(train_dataloader, model, loss_fn, optimizer)
  test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.284213 [    0/60000]
loss: 0.561023 [ 6400/60000]
loss: 0.379982 [12800/60000]
loss: 0.486144 [19200/60000]
loss: 0.456404 [25600/60000]
loss: 0.450402 [32000/60000]
loss: 0.388884 [38400/60000]
loss: 0.530139 [44800/60000]
loss: 0.468013 [51200/60000]
loss: 0.536864 [57600/60000]
Test Error: 
 Accuracy: 85.0%, Avg loss: 0.416069 

Epoch 2
-------------------------------
loss: 0.278202 [    0/60000]
loss: 0.370813 [ 6400/60000]
loss: 0.261259 [12800/60000]
loss: 0.382652 [19200/60000]
loss: 0.399119 [25600/60000]
loss: 0.378766 [32000/60000]
loss: 0.319344 [38400/60000]
loss: 0.499451 [44800/60000]
loss: 0.382705 [51200/60000]
loss: 0.479489 [57600/60000]
Test Error: 
 Accuracy: 85.5%, Avg loss: 0.390898 

Epoch 3
-------------------------------
loss: 0.220000 [    0/60000]
loss: 0.350737 [ 6400/60000]
loss: 0.226696 [12800/60000]
loss: 0.310688 [19200/60000]
loss: 0.361768 [25600/60000]
loss: 0.351552 [32000/60000]
loss: 0.298747 [38400/

In [ ]:
# Save the model
torch.save(model.state_dict(), 'fashionmnist_model.pth')

print('Model trained and saved.')

Model trained and saved.
